# DocuChat RAG - Google Colab Walkthrough

This notebook is a full, step-by-step environment to explore and test the repo line by line in Google Colab.

What you can do here:
- Install dependencies
- Load repo files and inspect them with line numbers
- Test each service module independently
- Run ingestion + retrieval checks
- Run Groq API smoke tests
- Optionally launch Streamlit from Colab with an ngrok tunnel

Before running: set your GitHub repo URL and Groq API key in the setup cells.

In [ ]:
#@title 1) Clone Repo
# Set your repository URL (HTTPS)
REPO_URL = "https://github.com/your-username/docuchat-rag.git"
REPO_DIR = "/content/docuchat-rag"

import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already exists, pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
#@title 2) Install Dependencies for Colab
# Colab uses pip, not uv.
# This includes your app dependencies plus tools for notebook testing and optional tunneling.
import sys

!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q \
  streamlit>=1.35 \
  langchain>=0.3 \
  langchain-community>=0.3 \
  langchain-huggingface>=0.1 \
  langchain-chroma>=0.1 \
  langchain-groq>=0.1 \
  chromadb>=0.5 \
  groq>=0.9 \
  sentence-transformers>=3.0 \
  pypdf>=4.0 \
  docx2txt>=0.8 \
  unstructured>=0.14 \
  python-dotenv>=1.0 \
  torch>=2.2 \
  transformers>=4.40 \
  pyngrok>=7.0

print("Dependency installation complete.")

In [ ]:
#@title 3) Configure Secrets (GROQ_API_KEY, optional HF_TOKEN)
# Preferred in Colab: store secret in Secrets (key icon) and use userdata.get().
# Fallback: type directly in this cell (not recommended for shared notebooks).

import os

try:
    from google.colab import userdata
    groq_key = userdata.get("GROQ_API_KEY")
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    groq_key = None
    hf_token = None

if not groq_key:
    groq_key = ""  # paste key if not using userdata
if not hf_token:
    hf_token = ""  # optional

if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

print("GROQ_API_KEY set:", bool(os.environ.get("GROQ_API_KEY")))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
#@title 4) Quick Project Tree
import os

def print_tree(root, max_depth=2, prefix=""):
    root = os.path.abspath(root)
    base_depth = root.rstrip(os.sep).count(os.sep)
    for current, dirs, files in os.walk(root):
        depth = current.rstrip(os.sep).count(os.sep) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(current)}/")
        for f in sorted(files):
            print(f"{indent}  {f}")

print_tree(".", max_depth=2)

In [ ]:
#@title 5) Helper: View Files with Line Numbers
from pathlib import Path

def show_file(path, start=1, end=200):
    p = Path(path)
    if not p.exists():
        print(f"File not found: {path}")
        return

    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    end = min(end, len(lines))
    for i in range(start, end + 1):
        print(f"{i:4d}: {lines[i-1]}")

print("Helper ready. Example: show_file('app.py', 1, 120)")

## Code Walkthrough
Run the next cells in order to inspect each source file line by line.

In [ ]:
#@title 6) Inspect app.py
show_file('app.py', 1, 260)

In [ ]:
#@title 7) Inspect services/config.py
show_file('services/config.py', 1, 260)

In [ ]:
#@title 8) Inspect services/embeddings.py
show_file('services/embeddings.py', 1, 220)

In [ ]:
#@title 9) Inspect services/vectorstore.py
show_file('services/vectorstore.py', 1, 260)

In [ ]:
#@title 10) Inspect services/ingestion.py
show_file('services/ingestion.py', 1, 320)

In [ ]:
#@title 11) Inspect services/llm.py
show_file('services/llm.py', 1, 240)

## Runtime and Service Smoke Tests

In [ ]:
#@title 12) Test runtime config and secret resolution
import os
from services.config import configure_runtime, get_secret

configure_runtime()
print('GROQ_API_KEY available:', bool(get_secret('GROQ_API_KEY')))
print('HF_TOKEN available:', bool(get_secret('HF_TOKEN')))

In [ ]:
#@title 13) Groq non-streaming smoke test
from services.llm import _get_client

resp = _get_client().chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'Reply with exactly: Groq API works.'}],
    max_tokens=10,
    temperature=0
)
print(resp.choices[0].message.content)

In [ ]:
#@title 14) Groq streaming path smoke test (matches app behavior)
from services.llm import stream_answer

text = ''.join(stream_answer('Reply with exactly: streaming works.', 'Context: test context'))
print(text)

In [ ]:
#@title 15) Embedding model initialization test
# First run may download model files and take a few minutes.
from services.embeddings import get_embeddings

model = get_embeddings()
print(type(model).__name__)

In [ ]:
#@title 16) Ingestion + retrieval integration test with a synthetic text file
import io
from services.ingestion import ingest_file, clear_all_documents
from services.vectorstore import similarity_search

clear_all_documents()

sample_text = (
    'Paris is the capital of France. '
    'Berlin is the capital of Germany. '
    'Tokyo is the capital of Japan.'
)

class UploadedLike(io.BytesIO):
    def __init__(self, content: str, name: str):
        super().__init__(content.encode('utf-8'))
        self.name = name

fake_upload = UploadedLike(sample_text, 'sample.txt')
chunks = ingest_file(fake_upload)
print('Chunks indexed:', chunks)

hits = similarity_search('What is the capital of France?', k=4, threshold=0.0)
print('Hits:', len(hits))
for h in hits[:2]:
    print('-', h['source'], 'chunk', h['chunk'], 'score', h['score'])
    print('  snippet:', h['snippet'])

## Optional: Run Streamlit App in Colab
This section launches the app and exposes it via an ngrok URL.

In [ ]:
#@title 17) (Optional) Set ngrok auth token
# Create at: https://dashboard.ngrok.com/get-started/your-authtoken
# In Colab Secrets, store as NGROK_AUTHTOKEN.
import os

try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    ngrok_token = ''

if ngrok_token:
    os.environ['NGROK_AUTHTOKEN'] = ngrok_token

print('NGROK_AUTHTOKEN set:', bool(os.environ.get('NGROK_AUTHTOKEN')))

In [ ]:
#@title 18) (Optional) Launch Streamlit + Open Tunnel
import os
import subprocess
import time
from pyngrok import ngrok

if os.environ.get('NGROK_AUTHTOKEN'):
    ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])

proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(4)
public_url = ngrok.connect(8501, bind_tls=True).public_url
print('Streamlit public URL:', public_url)
print('Process PID:', proc.pid)

In [ ]:
#@title 19) (Optional) Stop Streamlit/ngrok
from pyngrok import ngrok

try:
    proc.terminate()
    print('Stopped Streamlit process.')
except Exception:
    print('No running Streamlit process handle found.')

try:
    ngrok.kill()
    print('Stopped ngrok.')
except Exception:
    print('No active ngrok tunnel found.')

## Notes
- If Groq tests pass here but fail in app usage, restart the runtime and rerun setup cells.
- First embedding test is slower due to model download.
- You can delete this notebook or any test scripts after debugging.